# Feature Extraction Evaluation Pipeline

This notebook implements a complete workflow to test and evaluate LLM-based feature extraction against manually annotated ecological dataset metadata.

## Workflow Steps

### 1. Data Loading & Filtering
- Load the manually annotated dataset (`dataset_092624.xlsx`)
- Filter for Dryad source records with `valid_yn='yes'`
- Select rows where `_position` columns contain 'abstract' (indicating features were found in abstract)
- Select 5 test rows for evaluation

### 2. Data Validation
- Validate the manual annotations against the `DatasetFeatureExtraction` Pydantic schema
- Clean and normalize data using the `DataFrameValidator`

### 3. Automated Feature Extraction
- Extract abstracts from the `full_text` column
- Run GPT-based classification using `DatasetFeatureExtraction` schema
- Collect extraction results

### 4. Side-by-Side Comparison
- Build a comparison DataFrame with manual vs automated extractions
- Display results for visual inspection

### 5. Evaluation & Metrics
- Use `evaluation.py` utilities to compute precision, recall, F1 scores
- Normalize data between manual and automated sets as needed
- Generate evaluation report

---
**Test Dataset**: 5 Dryad records with abstract-derived annotations

### 1. Data Loading & Filtering


In [1]:
# Data exploration: load and inspect the dataset
import pandas as pd
import os

print("Current working directory:", os.getcwd())

# Use absolute path to avoid issues
ANNOT_FILE = r"data\dataset_092624.xlsx"
df = pd.read_excel(ANNOT_FILE)

print("Shape:", df.shape)
print("\nAll columns:")
print(df.columns.tolist())

Current working directory: c:\Users\beav3503\dev\llm_metadata
Shape: (418, 43)

All columns:
['id', 'url', 'title', 'full_text', 'publication_year', 'source', 'id_query', 'reason_non_valid', 'valid_yn', 'dataset_relevance', 'dataset_location', 'dataset_format', 'geospatial_info_dataset', 'geospatial_info_repo_page_text', 'geospatial_info_article_text', 'species', 'temporal_range', 'temp_range_i', 'temp_range_f', 'temporal_range_norm', 'temporal_range_position', 'temporal_duration_y', 'temporal_duration_position', 'spatial_range_km2', 'spatial_range_position', 'data_type', 'data_type_norm', 'referred_dataset', 'relevance_referred_dataset', 'journal', 'cited_articles', 'time_series', 'multispecies', 'threatened_species', 'new_species_science', 'new_species_region', 'bias_north_south', 'url.1', 'MC_dataset_type', 'MC_spatial_range', 'MC_temporal_range', 'MC_relevance', 'MC_relevance_modifiers']


In [2]:
# Explore _position columns and their values
position_cols = [c for c in df.columns if '_position' in c.lower()]
print("Position columns:", position_cols)

for col in position_cols:
    print(f"\n{col} unique values:")
    print(df[col].value_counts(dropna=False))

Position columns: ['temporal_range_position', 'temporal_duration_position', 'spatial_range_position']

temporal_range_position unique values:
temporal_range_position
NaN                                                       281
source publication text                                    86
source publication text, dataset                           12
not given                                                   9
source link abstract                                        7
source link abstract, dataset                               5
source link abstract, source publication text, dataset      4
source link                                                 4
source link abstract, source publication text               4
dataset                                                     2
source link abstract, dataset, source publication text      1
source link, dataset                                        1
dataset, source link                                        1
dataset, source publication 

In [3]:
# Filter: source='dryad', valid_yn='yes', and at least one _position column contains 'abstract'
print("Source values:", df['source'].value_counts())
print("\nvalid_yn values:", df['valid_yn'].value_counts())

# Filter for dryad and valid
dryad_valid = df[(df['source'] == 'dryad') & (df['valid_yn'] == 'yes')]
print(f"\nDryad valid rows: {len(dryad_valid)}")

# Find rows where any _position column contains 'abstract'
position_cols = ['temporal_range_position', 'temporal_duration_position', 'spatial_range_position']

def has_abstract_in_position(row):
    for col in position_cols:
        val = row.get(col)
        if pd.notna(val) and 'abstract' in str(val).lower():
            return True
    return False

dryad_abstract = dryad_valid[dryad_valid.apply(has_abstract_in_position, axis=1)]
print(f"Dryad valid rows with 'abstract' in _position columns: {len(dryad_abstract)}")

Source values: source
semantic_scholar    254
zenodo              114
dryad                47
referenced            3
Name: count, dtype: int64

valid_yn values: valid_yn
yes    299
no     119
Name: count, dtype: int64

Dryad valid rows: 37
Dryad valid rows with 'abstract' in _position columns: 11


In [4]:
# Check if there's an abstract column (full_text might be the abstract)
print("Checking full_text column (likely the abstract):")
print(f"Non-null full_text in dryad_abstract: {dryad_abstract['full_text'].notna().sum()}")
print("\nSample full_text (first 500 chars):")
print(dryad_abstract['full_text'].iloc[0][:500] if pd.notna(dryad_abstract['full_text'].iloc[0]) else "N/A")

Checking full_text column (likely the abstract):
Non-null full_text in dryad_abstract: 11

Sample full_text (first 500 chars):
Data from: The genetic signature of range expansion in a disease vector - the black-legged tick  Monitoring and predicting the spread of emerging infectious diseases requires that we understand the mechanisms of range expansion by its vectors. Here, we examined spatial and temporal variation of genetic structure among 13 populations of the Lyme disease vector, the black-legged tick, in southern Quebec, where this tick species is currently expanding and Lyme disease is emerging. Our objective was


In [5]:
dryad_abstract['url'].iloc[0]

'https://doi.org/10.5061/dryad.2n5h6'

In [6]:
# Show the relevant manual annotation columns that we'll compare with extraction
# These are features we'll extract and evaluate
relevant_cols = ['url', 'title', 'full_text', 'data_type', 'geospatial_info_dataset', 
                 'spatial_range_km2', 'temporal_range', 'temp_range_i', 'temp_range_f', 
                 'species', 'referred_dataset']

# Select 5 rows for testing
# For reproducibility, reuse same 5 from url

test_df = dryad_abstract.loc[[11, 15, 17, 18, 24], relevant_cols]

print(f"Selected {len(test_df)} rows for testing")
print("\nTest rows preview:")
test_df[['url', 'title', 'data_type', 'species']].head()

Selected 5 rows for testing

Test rows preview:


,url,title,data_type,species
11,https://doi.org/10.5061/dryad.2n5h6,Data from: The genetic signature of range expa...,"presence only, EBV genetic analysis",black-legged tick
15,https://doi.org/10.5061/dryad.3nh72,Data from: Patchy distribution and low effecti...,"presence only, EBV genetic analysis","Eastern coyote, eastern wolf"
17,https://doi.org/10.5061/dryad.4767v,Data from: Ecological gradients driving the di...,abundance,"Rhododendron groenlandicum, Kalmia angustifoli..."
18,https://doi.org/10.5061/dryad.4k275,"Data from: Caribou, water, and ice – fine-scal...",presence only,Rangifer tarandus
24,https://doi.org/10.5061/dryad.67t23,Data from: Severe recent decrease of adult bod...,EBV genetic analysis,Tachycineta bicolor


## Step 2: Validate Manual Annotations

In [7]:
%load_ext autoreload
%autoreload 2

In [8]:
from llm_metadata.schemas.validation import DataFrameValidator
from llm_metadata.schemas.fuster_features import DatasetFeatureExtraction

# Validate the test dataframe against the Pydantic schema
validator = DataFrameValidator(DatasetFeatureExtraction)
report = validator.validate(test_df)

print(report.summary())

if report.errors:
    print("\nSample Errors:")
    display(report.errors_to_dataframe())

2026-01-07 14:41:18.680 | INFO     | llm_metadata.schemas.validation:validate:233 - Validation complete: 5 valid, 0 invalid, 0 errors


VALIDATION REPORT
Total rows:     5
Valid rows:     5 (100.0%)
Invalid rows:   0

Error Breakdown:
  Type/Schema errors: 0
  Data/Value errors:  0

Errors by Field:


In [9]:
# Get validated manual annotations as Pydantic models
manual_models = report.valid_rows
manual_indices = report.valid_indices

# Create a mapping of DOI -> manual model for evaluation later
def extract_doi(url):
    return url.replace('https://doi.org/', '')

manual_by_doi = {}
for idx, model in zip(manual_indices, manual_models):
    doi = extract_doi(test_df.loc[idx, 'url'])
    manual_by_doi[doi] = model

print(f"Validated {len(manual_by_doi)} manual annotations")
print("DOIs:", list(manual_by_doi.keys()))

Validated 5 manual annotations
DOIs: ['10.5061/dryad.2n5h6', '10.5061/dryad.3nh72', '10.5061/dryad.4767v', '10.5061/dryad.4k275', '10.5061/dryad.67t23']


## Step 3: Automated Feature Extraction

Run GPT-based feature extraction on the abstracts using the `DatasetFeatureExtraction` schema.

In [10]:
from llm_metadata.gpt_classify import classify_abstract
from llm_metadata.schemas.fuster_features import DatasetFeatureExtraction

# Extract features from abstracts
automated_by_doi = {}

for idx in manual_indices:
    row = test_df.loc[idx]
    doi = extract_doi(row['url'])
    abstract = row['full_text']
    
    print(f"Processing DOI: {doi}")
    
    # Run GPT classification with DatasetFeatureExtraction schema
    result = classify_abstract(
        abstract=abstract,
        text_format=DatasetFeatureExtraction,
        model="gpt-5-mini",
        reasoning={"effort": "low"},
        max_output_tokens=4096,
    )
    
    automated_by_doi[doi] = result['output']
    print(f"  -> Extracted: {result['output'].data_type}")

print(f"\nCompleted extraction for {len(automated_by_doi)} abstracts")

Processing DOI: 10.5061/dryad.2n5h6
  -> Extracted: ['genetic_analysis', 'time_series', 'distribution']
Processing DOI: 10.5061/dryad.3nh72
  -> Extracted: ['genetic_analysis', 'distribution']
Processing DOI: 10.5061/dryad.4767v
  -> Extracted: ['abundance', 'distribution']
Processing DOI: 10.5061/dryad.4k275
  -> Extracted: ['presence-only', 'distribution', 'time_series']
Processing DOI: 10.5061/dryad.67t23
  -> Extracted: ['traits', 'abundance', 'time_series', 'ecosystem_function', 'distribution']

Completed extraction for 5 abstracts


## Step 4: Vocabulary Normalization

Normalize vocabulary between manual annotations and automated extractions to enable fair comparison.
- Map free-text `data_type` values to `EBVDataType` enum
- Map free-text `geospatial_info_dataset` values to `GeospatialInfoType` enum
- Apply fuzzy matching for `species` field

In [15]:
# Vocabulary normalization mappings
from llm_metadata.schemas.fuster_features import EBVDataType, GeospatialInfoType
from rapidfuzz import fuzz

# Manual annotation -> EBVDataType mapping
DATA_TYPE_MAPPING = {
    # Direct matches
    "abundance": EBVDataType.ABUNDANCE,
    "presence-absence": EBVDataType.PRESENCE_ABSENCE,
    "presence only": EBVDataType.PRESENCE_ONLY,
    "presence-only": EBVDataType.PRESENCE_ONLY,
    "density": EBVDataType.DENSITY,
    "distribution": EBVDataType.DISTRIBUTION,
    "traits": EBVDataType.TRAITS,
    "ecosystem function": EBVDataType.ECOSYSTEM_FUNCTION,
    "ecosystem_function": EBVDataType.ECOSYSTEM_FUNCTION,
    "ecosystem structure": EBVDataType.ECOSYSTEM_STRUCTURE,
    "ecosystem_structure": EBVDataType.ECOSYSTEM_STRUCTURE,
    "genetic analysis": EBVDataType.GENETIC_ANALYSIS,
    "genetic_analysis": EBVDataType.GENETIC_ANALYSIS,
    "ebv genetic analysis": EBVDataType.GENETIC_ANALYSIS,
    "time series": EBVDataType.TIME_SERIES,
    "time_series": EBVDataType.TIME_SERIES,
    "species richness": EBVDataType.SPECIES_RICHNESS,
    "species_richness": EBVDataType.SPECIES_RICHNESS,
    "other": EBVDataType.OTHER,
    "unknown": EBVDataType.UNKNOWN,
}

# Manual annotation -> GeospatialInfoType mapping
GEO_TYPE_MAPPING = {
    "sample": GeospatialInfoType.SAMPLE,
    "sample coordinates": GeospatialInfoType.SAMPLE,
    "site": GeospatialInfoType.SITE,
    "site coordinates": GeospatialInfoType.SITE,
    "range": GeospatialInfoType.RANGE,
    "range coordinates": GeospatialInfoType.RANGE,
    "distribution": GeospatialInfoType.DISTRIBUTION,
    "species distribution": GeospatialInfoType.DISTRIBUTION,
    "geographic features": GeospatialInfoType.GEOGRAPHIC_FEATURES,
    "geographic_features": GeospatialInfoType.GEOGRAPHIC_FEATURES,
    "administrative units": GeospatialInfoType.ADMINISTRATIVE_UNITS,
    "administrative_units": GeospatialInfoType.ADMINISTRATIVE_UNITS,
    "maps": GeospatialInfoType.MAPS,
    "site ids": GeospatialInfoType.SITE_IDS,
    "site_ids": GeospatialInfoType.SITE_IDS,
    "unknown": GeospatialInfoType.UNKNOWN,
}


def normalize_data_type(values: list | None) -> list[EBVDataType] | None:
    """Normalize data_type values to EBVDataType enum."""
    if values is None:
        return None
    
    normalized = []
    for val in values:
        if isinstance(val, EBVDataType):
            normalized.append(val)
        elif isinstance(val, str):
            key = val.lower().strip()
            if key in DATA_TYPE_MAPPING:
                normalized.append(DATA_TYPE_MAPPING[key])
            else:
                # Fuzzy match for typos
                best_match = None
                best_score = 0
                for k, v in DATA_TYPE_MAPPING.items():
                    score = fuzz.ratio(key, k)
                    if score > best_score and score > 70:
                        best_score = score
                        best_match = v
                if best_match:
                    normalized.append(best_match)
                else:
                    print(f"  Warning: Unknown data_type '{val}' - skipping")
    
    return list(set(normalized)) if normalized else None


def normalize_geo_type(values: list | None) -> list[GeospatialInfoType] | None:
    """Normalize geospatial_info_dataset values to GeospatialInfoType enum."""
    if values is None:
        return None
    
    normalized = []
    for val in values:
        if isinstance(val, GeospatialInfoType):
            normalized.append(val)
        elif isinstance(val, str):
            key = val.lower().strip()
            if key in GEO_TYPE_MAPPING:
                normalized.append(GEO_TYPE_MAPPING[key])
            else:
                # Fuzzy match
                best_match = None
                best_score = 0
                for k, v in GEO_TYPE_MAPPING.items():
                    score = fuzz.ratio(key, k)
                    if score > best_score and score > 70:
                        best_score = score
                        best_match = v
                if best_match:
                    normalized.append(best_match)
                else:
                    print(f"  Warning: Unknown geospatial_info '{val}' - skipping")
    
    return list(set(normalized)) if normalized else None


def normalize_species_fuzzy(manual_species: list | None, auto_species: list | None, threshold: int = 70) -> tuple[set, set]:
    """
    Fuzzy match species lists and return normalized sets.
    
    Returns (manual_normalized, auto_normalized) where matching species use the same string.
    """
    if manual_species is None:
        manual_species = []
    if auto_species is None:
        auto_species = []
    
    manual_normalized = set()
    auto_normalized = set()
    
    # Create mapping from auto to manual based on fuzzy matching
    auto_to_manual = {}
    
    for auto_sp in auto_species:
        auto_lower = auto_sp.lower().strip()
        best_match = None
        best_score = 0
        
        for manual_sp in manual_species:
            manual_lower = manual_sp.lower().strip()
            score = fuzz.ratio(auto_lower, manual_lower)
            if score > best_score:
                best_score = score
                if score >= threshold:
                    best_match = manual_sp
        
        if best_match:
            auto_to_manual[auto_sp] = best_match
            auto_normalized.add(best_match.lower())
        else:
            auto_normalized.add(auto_lower)
    
    # Add all manual species (normalized)
    for manual_sp in manual_species:
        manual_normalized.add(manual_sp.lower().strip())
    
    return manual_normalized, auto_normalized


print("Vocabulary normalization functions defined.")

Vocabulary normalization functions defined.


In [16]:
# Apply normalization to both manual and automated extractions
from llm_metadata.schemas.fuster_features import DatasetFeatureExtraction

def normalize_extraction(model: DatasetFeatureExtraction) -> DatasetFeatureExtraction:
    """Create a normalized copy of the extraction model."""
    data = model.model_dump()
    
    # Normalize data_type
    data['data_type'] = normalize_data_type(data.get('data_type'))
    
    # Normalize geospatial_info_dataset
    data['geospatial_info_dataset'] = normalize_geo_type(data.get('geospatial_info_dataset'))
    
    return DatasetFeatureExtraction.model_validate(data)


# Create normalized copies
manual_normalized = {}
auto_normalized = {}

for doi in manual_by_doi.keys():
    print(f"\nNormalizing DOI: {doi}")
    
    manual = manual_by_doi[doi]
    auto = automated_by_doi[doi]
    
    # Normalize enum fields
    print("  Manual data_type:", manual.data_type)
    manual_norm = normalize_extraction(manual)
    print("  -> Normalized:", manual_norm.data_type)
    
    print("  Auto data_type:", auto.data_type)
    auto_norm = normalize_extraction(auto)
    print("  -> Normalized:", auto_norm.data_type)
    
    # Handle species fuzzy matching - modify the normalized data
    manual_species_set, auto_species_set = normalize_species_fuzzy(
        manual.species, auto.species, threshold=70
    )
    
    # Update the species field with normalized values
    manual_data = manual_norm.model_dump()
    auto_data = auto_norm.model_dump()
    
    manual_data['species'] = list(manual_species_set) if manual_species_set else None
    auto_data['species'] = list(auto_species_set) if auto_species_set else None
    
    manual_normalized[doi] = DatasetFeatureExtraction.model_validate(manual_data)
    auto_normalized[doi] = DatasetFeatureExtraction.model_validate(auto_data)
    
    print(f"  Species: manual={len(manual_species_set)} auto={len(auto_species_set)}")

print(f"\n✓ Normalized {len(manual_normalized)} records")


Normalizing DOI: 10.5061/dryad.2n5h6
  Manual data_type: ['presence-only', 'genetic_analysis']
  -> Normalized: ['genetic_analysis', 'presence-only']
  Auto data_type: ['genetic_analysis', 'time_series', 'distribution']
  -> Normalized: ['time_series', 'genetic_analysis', 'distribution']
  Species: manual=1 auto=4

Normalizing DOI: 10.5061/dryad.3nh72
  Manual data_type: ['presence-only', 'genetic_analysis']
  -> Normalized: ['genetic_analysis', 'presence-only']
  Auto data_type: ['genetic_analysis', 'distribution']
  -> Normalized: ['genetic_analysis', 'distribution']
  Species: manual=2 auto=11

Normalizing DOI: 10.5061/dryad.4767v
  Manual data_type: ['abundance']
  -> Normalized: ['abundance']
  Auto data_type: ['abundance', 'distribution']
  -> Normalized: ['abundance', 'distribution']
  Species: manual=4 auto=6

Normalizing DOI: 10.5061/dryad.4k275
  Manual data_type: ['presence-only']
  -> Normalized: ['presence-only']
  Auto data_type: ['presence-only', 'distribution', 'time_s

## Step 5: Side-by-Side Comparison

Create a comparison DataFrame showing manual vs automated extractions for visual inspection.
**Note:** Dropped `temporal_range` (free text) from evaluation, but kept `temp_range_i` and `temp_range_f` (year integers).

In [21]:
# Build side-by-side comparison DataFrame (using normalized data)
# Keep temp_range_i and temp_range_f, but drop temporal_range (free text)
comparison_fields = ['data_type', 'geospatial_info_dataset', 'spatial_range_km2', 
                     'temp_range_i', 'temp_range_f', 'species']

comparison_rows = []

for doi in manual_normalized.keys():
    manual = manual_normalized[doi]
    auto = auto_normalized[doi]
    
    # Add manual row
    manual_dict = manual.model_dump()
    manual_row = {'doi': doi, 'source': 'manual'}
    for field in comparison_fields:
        manual_row[field] = manual_dict.get(field)
    comparison_rows.append(manual_row)
    
    # Add automated row
    auto_dict = auto.model_dump()
    auto_row = {'doi': doi, 'source': 'automated'}
    for field in comparison_fields:
        auto_row[field] = auto_dict.get(field)
    comparison_rows.append(auto_row)

comparison_df = pd.DataFrame(comparison_rows)
comparison_df = comparison_df.sort_values(['doi', 'source'], ascending=[True, False])
comparison_df

,doi,source,data_type,geospatial_info_dataset,spatial_range_km2,temp_range_i,temp_range_f,species
0,10.5061/dryad.2n5h6,manual,"[genetic_analysis, presence-only]",[sample],2080.0,2011.0,2014.0,[black-legged tick]
1,10.5061/dryad.2n5h6,automated,"[time_series, genetic_analysis, distribution]","[sample, administrative_units, site]",NaN,2011.0,2014.0,"[ixodes scapularis, lyme disease vector, 613 t..."
2,10.5061/dryad.3nh72,manual,"[genetic_analysis, presence-only]",[sample],NaN,2010.0,2014.0,"[eastern coyote, eastern wolf]"
3,10.5061/dryad.3nh72,automated,"[genetic_analysis, distribution]","[sample, administrative_units, geographic_feat...",NaN,2010.0,2014.0,"[eastern wolf, canis individuals, quebec, cani..."
4,10.5061/dryad.4767v,manual,[abundance],None,535355.0,1986.0,2000.0,"[vaccinium spp, chamaedaphne calyculata, kalmi..."
5,10.5061/dryad.4767v,automated,"[abundance, distribution]","[sample, administrative_units, maps, range, di...",535355.0,NaN,NaN,"[kalmia angustifolia, vaccinium angustifolium,..."
6,10.5061/dryad.4k275,manual,[presence-only],[sample],630000.0,1947.0,1985.0,[rangifer tarandus]
7,10.5061/dryad.4k275,automated,"[time_series, distribution, presence-only]","[sample, maps, geographic_features]",NaN,1947.0,2014.0,"[rangifer tarandus, migratory caribou]"
8,10.5061/dryad.67t23,manual,[genetic_analysis],[site],10200.0,2005.0,2011.0,[tachycineta bicolor]
9,10.5061/dryad.67t23,automated,"[time_series, traits, ecosystem_function, abun...",[administrative_units],NaN,2005.0,2011.0,"[tachycineta bicolor, aerial insectivores]"


## Step 6: Evaluation & Metrics

Use the `evaluation.py` utilities to compute precision, recall, F1, and other metrics on normalized data.
**Note:** Dropped `temporal_range` (free text), kept `temp_range_i` and `temp_range_f` (year integers).

In [22]:
from llm_metadata.schemas.evaluation import (
    evaluate_indexed,
    EvaluationConfig,
    micro_average,
    macro_f1
)

# Run evaluation using NORMALIZED dictionaries
# Dropped temporal_range (free text), kept temp_range_i and temp_range_f (year integers)
eval_fields = ['data_type', 'geospatial_info_dataset', 'spatial_range_km2', 
               'temp_range_i', 'temp_range_f', 'species']

report = evaluate_indexed(
    true_by_id=manual_normalized,
    pred_by_id=auto_normalized,
    fields=eval_fields,
    config=EvaluationConfig(treat_lists_as_sets=True)
)

print("Per-field metrics (with vocabulary normalization):")
display(report.metrics_to_pandas())

Per-field metrics (with vocabulary normalization):


,field,tp,fp,fn,tn,n,precision,recall,f1,accuracy,exact_match_rate
0,data_type,4,11,3,0,5,0.266667,0.571429,0.363636,0.8,0.0
1,geospatial_info_dataset,3,14,1,0,5,0.176471,0.750000,0.285714,0.6,0.0
2,spatial_range_km2,1,0,3,1,5,1.000000,0.250000,0.400000,0.4,0.4
5,species,9,16,0,0,5,0.360000,1.000000,0.529412,1.8,0.0
4,temp_range_f,3,1,2,0,5,0.750000,0.600000,0.666667,0.6,0.6
3,temp_range_i,4,0,1,0,5,1.000000,0.800000,0.888889,0.8,0.8


In [23]:
# Aggregate metrics
micro = micro_average(report.field_metrics.values())
macro = macro_f1(report.field_metrics.values())

print("=" * 50)
print("AGGREGATE METRICS")
print("=" * 50)
print(f"Micro-average Precision: {micro.precision:.3f}" if micro.precision else "Micro Precision: N/A")
print(f"Micro-average Recall:    {micro.recall:.3f}" if micro.recall else "Micro Recall: N/A")
print(f"Micro-average F1:        {micro.f1:.3f}" if micro.f1 else "Micro F1: N/A")
print(f"Macro-average F1:        {macro:.3f}" if macro else "Macro F1: N/A")
print("=" * 50)

AGGREGATE METRICS
Micro-average Precision: 0.364
Micro-average Recall:    0.706
Micro-average F1:        0.480
Macro-average F1:        0.522


In [24]:
# Detailed per-record comparison results
print("Per-record, per-field results (mismatches highlighted):")
results_df = report.to_pandas()
mismatches = results_df[~results_df['match']]
print(f"\nTotal mismatches: {len(mismatches)}")
display(mismatches)

Per-record, per-field results (mismatches highlighted):

Total mismatches: 21


,record_id,field,true_value,pred_value,match,tp,fp,fn,tn
0,10.5061/dryad.2n5h6,data_type,"[genetic_analysis, presence-only]","[time_series, genetic_analysis, distribution]",False,1,2,1,0
1,10.5061/dryad.2n5h6,geospatial_info_dataset,[sample],"[sample, administrative_units, site]",False,1,2,0,0
2,10.5061/dryad.2n5h6,spatial_range_km2,2080.0,None,False,0,0,1,0
5,10.5061/dryad.2n5h6,species,[black-legged tick],"[ixodes scapularis, lyme disease vector, 613 t...",False,1,3,0,0
6,10.5061/dryad.3nh72,data_type,"[genetic_analysis, presence-only]","[genetic_analysis, distribution]",False,1,1,1,0
7,10.5061/dryad.3nh72,geospatial_info_dataset,[sample],"[sample, administrative_units, geographic_feat...",False,1,4,0,0
11,10.5061/dryad.3nh72,species,"[eastern coyote, eastern wolf]","[eastern wolf, canis individuals, quebec, cani...",False,2,9,0,0
12,10.5061/dryad.4767v,data_type,[abundance],"[abundance, distribution]",False,1,1,0,0
13,10.5061/dryad.4767v,geospatial_info_dataset,None,"[sample, administrative_units, maps, range, di...",False,0,5,0,0
15,10.5061/dryad.4767v,temp_range_i,1986,None,False,0,0,1,0


---

## Results Synthesis & Analysis

### Overall Performance (with Vocabulary Normalization)
| Metric | Value | Change from raw |
|--------|-------|-----------------|
| Micro-average Precision | 0.293 | -1% |
| Micro-average Recall | 0.708 | +32% ⬆️ |
| Micro-average F1 | 0.415 | +9% ⬆️ |
| Macro-average F1 | 0.395 | -20% |

**Note:** Dropped temporal fields (`temporal_range`, `temp_range_i`, `temp_range_f`) from evaluation.

### Per-Field Analysis

**Strong Performance:**
- **Species**: Recall = 1.0 (perfect!), Precision = 0.36, F1 = 0.53 — Fuzzy matching dramatically improved recall. The model now captures all manual species with some over-extraction.

**Moderate Performance:**
- **Spatial range** (`spatial_range_km2`): Precision = 1.0, Recall = 0.25, F1 = 0.40 — When the model extracts a value, it's correct, but it often misses values (conservative extraction).
- **Data type**: Precision = 0.27, Recall = 0.57, F1 = 0.36 — Model still over-extracts EBV categories (11 FP vs 4 TP). Vocabulary normalization didn't significantly change this.

**Weak Performance:**
- **Geospatial info**: Precision = 0.18, Recall = 0.75, F1 = 0.29 — Consistent over-prediction (14 FP). The model identifies geospatial concepts that annotators didn't explicitly mark.

### Key Observations

1. **Fuzzy matching works**: Species field recall jumped to 100% with fuzzy matching at threshold=70.

2. **Over-extraction persists**: Model systematically identifies more `data_type` and `geospatial_info_dataset` values than annotators. This could be:
   - Model correctly inferring from abstract text
   - Annotators using more restrictive interpretation
   - Schema definitions not aligned with annotation guidelines

3. **Spatial range is binary**: Either perfect match or miss — suggests the model is conservative about numeric extraction.

---

## Conclusion

With vocabulary normalization and fuzzy matching:
- **Micro F1 improved from 0.38 → 0.42** (+9%)
- **Recall jumped from 0.54 → 0.71** (+32%) — major improvement

The pipeline now properly handles vocabulary differences. Remaining issues are primarily **semantic** (model interprets abstracts differently than annotators) rather than **technical**.

---

## Next Steps (Updated)

### Completed ✓
1. ~~Standardize `data_type` vocabulary mapping~~ ✓
2. ~~Implement fuzzy matching for species~~ ✓
3. ~~Drop temporal fields from evaluation~~ ✓

### Next Priority
4. **Evidence extraction**: Add reasoning/excerpts to understand why model makes extraction decisions
5. **Expand test set**: Run on all 11 abstract-annotated Dryad records
6. **Prompt refinement**: Add few-shot examples aligned to annotation guidelines, especially for `data_type`

### Deferred
- Temporal field evaluation (decided not relevant for abstract-only extraction)
- Bootstrap confidence intervals (need larger test set first)

### Understanding the Metrics

**Core Metrics:**
- **Precision**: Of all values the model extracted, what fraction were correct? High precision = few false positives (model doesn't over-extract).
- **Recall**: Of all true values that exist, what fraction did the model find? High recall = few false negatives (model doesn't miss things).
- **F1 Score**: Harmonic mean of precision and recall (balances both). Best when both are high.

**Confusion Matrix Terms:**
- **TP (True Positives)**: Model correctly extracted values that match manual annotations
- **FP (False Positives)**: Model extracted values that don't match manual annotations (over-extraction)
- **FN (False Negatives)**: Manual annotations that the model missed (under-extraction)
- **TN (True Negatives)**: Cases where both agreed on absence (less relevant for extraction tasks)

**Aggregate Metrics:**
- **Micro-average**: Pools all TP/FP/FN across fields, then computes metrics (emphasizes high-volume fields)
- **Macro-average F1**: Averages F1 scores across all fields (treats each field equally, regardless of volume)

**Interpretation for This Task:**
- High precision, low recall → Model is conservative (extracts only when very confident)
- Low precision, high recall → Model is aggressive (extracts liberally but with errors)
- F1 near 0.40 suggests the model captures meaningful information but with substantial noise
- Field-specific F1 variance shows some features are easier to extract than others